# Neural-network robustness certification (auto_LiRPA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/fmaiv/blob/main/day04/examples/nn/robustness.ipynb)

**FMAIV Day 4 — the frontier hands-on.** Same *verification* question as the rest of the week
("can the bad thing happen?"), now for a neural network:

> Within an L-infinity ball of radius `eps` around an input `x0`, can the classifier's prediction change?

We use **auto_LiRPA** — the CROWN bound-propagation engine underneath
[alpha,beta-CROWN](https://github.com/Verified-Intelligence/alpha-beta-CROWN), the VNN-COMP winner.
It computes a **certified** lower bound on the margin `z[true] - z[other]` over the *entire* ball.
If that bound is `> 0`, **no** input in the ball is misclassified — a proof, not a sample.

Everything here is tiny and **CPU-only** (a 2-D, 2-class MLP), so it runs in ~1s on the free Colab
tier. The same code applies unchanged to a trained MNIST/CIFAR network — see our
[AAAI'26 VNN-COMP tutorial](https://vnn-comp.github.io/#aaai2026) for full-scale notebooks.


## 0. Install

On Colab `torch` is already present, so we only add `auto_LiRPA`. Its maintained
release lives on GitHub (PyPI is stuck at an ancient 0.2/0.3), pinned here for reproducibility.


In [ ]:
!pip -q install git+https://github.com/Verified-Intelligence/auto_LiRPA.git@ca767f1d8c0a6b125a292ba165adb2319bbaf615

## 1. A tiny dataset and model
Two well-separated Gaussian blobs (2 classes) and a small ReLU MLP. Deterministic via a fixed seed.


In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

def make_data(n=600):
    half = n // 2
    blob0 = torch.randn(half, 2) * 0.7 + torch.tensor([1.0, 1.0])   # class 0
    blob1 = torch.randn(half, 2) * 0.7 + torch.tensor([-1.0, -1.0]) # class 1 (some overlap)
    X = torch.cat([blob0, blob1], 0)
    y = torch.cat([torch.zeros(half), torch.ones(half)]).long()
    return X, y

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2,16), nn.ReLU(),
                                 nn.Linear(16,16), nn.ReLU(),
                                 nn.Linear(16,2))
    def forward(self, x):
        return self.net(x)

X, y = make_data()
model = MLP()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
lossf = nn.CrossEntropyLoss()
for _ in range(300):
    opt.zero_grad(); lossf(model(X), y).backward(); opt.step()
model.eval()
acc = (model(X).argmax(1) == y).float().mean().item()
print(f'train accuracy: {acc:.3f}')

## 2. Certify the margin under an L-inf perturbation
`compute_bounds(..., C=C, method='CROWN')` returns a certified lower bound on the linear
specification `C @ logits`. We set `C` to pick out the margin `z[true] - z[other]`.
A lower bound `> 0` means **certified robust** at that `eps`.


In [ ]:
from auto_LiRPA import BoundedModule, BoundedTensor
from auto_LiRPA.perturbations import PerturbationLpNorm

# A correctly-classified point with a moderate (not razor-thin) clean margin, so it
# certifies for a band of small eps and then breaks as the ball grows.
with torch.no_grad():
    logits = model(X)
    correct = logits.argmax(1) == y
    margins = torch.where(correct,
        logits.gather(1, y.view(-1,1)).squeeze(1) - logits.gather(1, (1-y).view(-1,1)).squeeze(1),
        torch.full_like(logits[:,0], float('inf')))
    idx = int((margins - 3.5).abs().argmin())
x0 = X[idx:idx+1]
true_cls = y[idx].item()
other = 1 - true_cls
lirpa_model = BoundedModule(model, torch.empty_like(x0))

C = torch.zeros(1, 1, 2)
C[0, 0, true_cls] = 1.0
C[0, 0, other] = -1.0

print(f'input = {[round(v,3) for v in x0.tolist()[0]]}, true class = {true_cls}')
print(f"{'eps':>6} {'cert. margin':>13}  verdict")
for eps in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    ptb = PerturbationLpNorm(norm=float('inf'), eps=eps)
    bx = BoundedTensor(x0, ptb)
    lb, ub = lirpa_model.compute_bounds(x=(bx,), C=C, method='CROWN')
    m = lb.item()
    verdict = 'CERTIFIED ROBUST' if m > 0 else 'not certified by CROWN'
    print(f'{eps:6.2f} {m:13.4f}  {verdict}')

## 3. Visualize the decision boundary and the eps-balls
The shaded regions are the network's prediction and the star is `x0`. Each **square** is an
L-infinity `eps`-ball around `x0` (side `2*eps`), drawn for several `eps` and **colored by the
CROWN verdict** from the table above: **green = certified robust** (the whole box provably stays
one color), **red = not certified**. The right panel **zooms in** on `x0` so the small balls are
actually visible — at the global scale a small `eps`-ball is only a few percent of the axes.
Robustness fails once a box grows far enough to touch the decision boundary.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from auto_LiRPA import BoundedTensor
from auto_LiRPA.perturbations import PerturbationLpNorm

# Decision regions over a grid (argmax class at each point).
xs = np.linspace(-3.5, 3.5, 300)
gx, gy = np.meshgrid(xs, xs)
grid = torch.tensor(np.stack([gx.ravel(), gy.ravel()], 1), dtype=torch.float32)
with torch.no_grad():
    pred = model(grid).argmax(1).numpy().reshape(gx.shape)

px, py = x0[0].tolist()
eps_show = [0.1, 0.25, 0.4]           # several nested L-inf balls, straddling the cert. threshold

# Color each box by whether CROWN *certifies* robustness at that eps
# (green = proven safe over the whole box; red = CROWN cannot certify) —
# the same verdicts as the table above, drawn on the picture.
def is_certified(eps):
    ptb = PerturbationLpNorm(norm=float('inf'), eps=eps)
    lb, _ = lirpa_model.compute_bounds(x=(BoundedTensor(x0, ptb),), C=C, method='CROWN')
    return lb.item() > 0
box_color = {e: ('#1a7f37' if is_certified(e) else '#c0392b') for e in eps_show}

def draw(ax):
    ax.contourf(gx, gy, pred, alpha=0.20, levels=1, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, s=7, cmap='coolwarm', alpha=0.45, zorder=1)
    for e in eps_show:                # boxes ON TOP, thick + colored, so they're visible
        ax.add_patch(Rectangle((px - e, py - e), 2*e, 2*e, fill=False,
                               edgecolor=box_color[e], lw=2.4, zorder=5))
    ax.scatter([px], [py], marker='*', s=170, edgecolor='k',
               facecolor='yellow', linewidths=1.0, zorder=6)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 5.2))

# Left: global view — where x0 sits relative to the decision boundary.
draw(axL)
axL.set_title('Decision regions + L-inf balls around x0')
axL.set_xlabel('x1'); axL.set_ylabel('x2')

# Right: zoom on x0 so the small certified boxes are clearly visible.
draw(axR)
pad = max(eps_show) + 0.3
axR.set_xlim(px - pad, px + pad); axR.set_ylim(py - pad, py + pad)
axR.set_title(f'Zoom on x0 = ({px:.2f}, {py:.2f})')
axR.set_xlabel('x1'); axR.set_ylabel('x2')

legend = [Line2D([0], [0], marker='*', color='w', markerfacecolor='yellow',
                 markeredgecolor='k', markersize=14, label='x0'),
          Line2D([0], [0], color='#1a7f37', lw=2.4, label='eps certified robust'),
          Line2D([0], [0], color='#c0392b', lw=2.4, label='eps not certified')]
axR.legend(handles=legend, loc='upper right', fontsize=8)
plt.suptitle(f'L-inf eps-balls at {eps_show}: green = proven safe, red = CROWN cannot certify')
plt.tight_layout(); plt.show()

## Takeaways
- **Certified `> 0`** = a *proof* of robustness over the whole ball (sound; like CBMC/Lean verdicts).
- **"not certified"** does NOT mean "vulnerable": CROWN is *incomplete*, so the bound can dip below 0
  while the point is still safe. Complete verifiers (alpha,beta-CROWN) close that gap with branch-and-bound.
- This is the same soundness/completeness story as the rest of the course, now for NNs.

**Next:** the script form is `robustness.py` (and `robustness_starter.py` blanks the `compute_bounds`
call). For full-scale MNIST/CIFAR verification and competition benchmarks, see the
[AAAI'26 VNN-COMP tutorial](https://vnn-comp.github.io/#aaai2026).
